In [0]:
import dlt
from pyspark.sql.functions import col, split, substring, length, regexp_replace, when, expr, cast, current_date, datediff,trim,regexp_extract
from pyspark.sql.types import DecimalType, IntegerType, DateType

In [0]:
env = spark.conf.get("pipeline.env") 
catalog       = "raiffka_catalog"
bronze_schema = f"bronze"
silver_schema = f"silver"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {silver_schema}")

In [0]:
#--------- CUSTOMERS ----------#
customers_expectations = {
    "valid_customer_id":"customer_id is not null",
    "valid_datum_narozeni":"year(datum_narozeni) > year(current_date())-100"
}

customer_expectations_expr = "NOT({0})".format(" AND ".join(customers_expectations.values()))

@dlt.table(
    name = "customers_clean_silver",
    comment = "This table contains the cleaned customers data",
    partition_cols =["is_quarantined"]
)
@dlt.expect_all(customers_expectations)
def customers_clean_silver():
    df_customer = dlt.readStream(f"{catalog}.{bronze_schema}.customers_bronze")
    df_base = (
        df_customer
        .withColumn("cela_adresa", col("adresa"))
        .withColumn("zip_code", split(col("adresa"), ",").getItem(2))
        .withColumn("mesto", split(col("adresa"), ",").getItem(1))
        .withColumn("ulice", split(col("adresa"), ",").getItem(0))
        .withColumn("datum_narozeni", col("datum_narozeni").cast(DateType()))
        .withColumn(
            "predcisli",
            when(
                length(regexp_replace(col("tel_cislo"), " ", "")) > 9,
                split(col("tel_cislo"), " ").getItem(0)
            ).otherwise(None)
        )
        .withColumn(
            "tel_cislo",
            substring(regexp_replace(col("tel_cislo"), " ", ""), -9, 9)
        )
        .withColumn("pocet_clenu_domacnost", col("pocet_clenu_domacnost").cast(IntegerType()))
        .withColumn("datum_prvniho_otevreni_uctu", col("datum_prvniho_otevreni_uctu").cast(DateType()))
        .withColumn("prijem", col("prijem").cast(DecimalType(20,2)))
        .withColumn("sum_prm_uspor_12m", col("sum_prm_uspor_12m").cast(DecimalType(20,2)))
        .withColumn("min_atm_amt", col("min_atm_amt").cast(DecimalType(20,2)))
        .withColumn("pocet_nemovitosti", col("pocet_nemovitosti").cast(IntegerType()))
    )
        
    return (
        df_base
        .withColumn("ulice", split(col("ulice"), " ").getItem(0))
        .withColumn("cislo_popisne", split(col("ulice"), " ").getItem(1))
        .withColumn("client_age", expr("year(current_date()) - year(datum_narozeni)"))
        .withColumn("days_from_open_first_acc", datediff(current_date(), col("datum_prvniho_otevreni_uctu")))
        .withColumn("is_quarantined",expr(customer_expectations_expr))
        .select(
            "customer_id",
            "jmeno",
            "prijmeni",
            "cela_adresa",
            "zip_code",
            "mesto",
            "ulice",
            "cislo_popisne",
            "email",
            "rodne_cislo",
            "stav",
            "predcisli",
            "tel_cislo",
            "datum_narozeni",
            "typ_bydleni",
            "pracovni_stav",
            "pohlavi",
            "pocet_clenu_domacnost",
            "datum_prvniho_otevreni_uctu",
            "zeme",
            "prijem",
            "titul_pred",
            "titul_za",
            "sum_prm_uspor_12m",
            "min_atm_amt",
            "rezident",
            "pocet_nemovitosti",
            "client_age",
            "days_from_open_first_acc",
            "snapshot_date",
            "is_quarantined"
        )
    )

@dlt.table(name='customers_clean_good_records',comment='customers cleanded and validate data')
def customer_clean():
    df_customer = dlt.readStream('customers_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=false")
        .drop("is_quarantined")
    )

@dlt.table(name='customers_clean_bad_records',comment='customers cleaned and bad data')
def customer_clean():
    df_customer = dlt.readStream('customers_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=true")
        .drop("is_quarantined")
    )

dlt.create_streaming_table(
        name="dim_customers",
        comment="cusotmers dim table with SCD2 approach",
        table_properties={"quality": "silver"}
        )
dlt.apply_changes(
        target = "dim_customers",
        source = "customers_clean_good_records",
        keys = ["customer_id"],
        sequence_by = col("snapshot_date"),
        ignore_null_updates=False,
        stored_as_scd_type="2",
        track_history_except_column_list=['snapshot_date']
    )


#------------------------------- ACCOUNT BALANCE -------------------------
@dlt.table(
    name = "account_balances_silver",
    comment = "This table contains the cleaned account balances data",
    table_properties={"quality": "silver"}
)
def account_balances_silver():
    df_ab = dlt.readStream(f"{catalog}.{bronze_schema}.account_balances_bronze")
    return (
        df_ab
        .withColumn("balance", col("balance").cast(DecimalType(20,2)))
        .withColumn("datum", col("datum").cast(DateType()))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- ACCOUNTS -------------------------
accounts_expectations = {
    "valid_account":"typ_uctu is not null and stav is not null",
    "valid_mena":"mena is not null",
}

accounts_expectations_expr = "NOT({0})".format(" AND ".join(accounts_expectations.values()))

@dlt.table(
    name = "accounts_clean_silver",
    comment = "This table contains the cleaned accounts data",
    partition_cols =["is_quarantined"]
)
@dlt.expect_all(accounts_expectations)
def accounts_clean_silver():
    df_accounts = dlt.readStream(f"{catalog}.{bronze_schema}.accounts_bronze")
    return (
        df_accounts
        .withColumn("is_quarantined",expr(accounts_expectations_expr))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

@dlt.table(name='accounts_clean_good_records',comment='accounts cleanded and validate data')
def customer_clean():
    df_customer = dlt.readStream('accounts_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=false")
        .drop("is_quarantined")
    )

@dlt.table(name='accounts_clean_bad_records',comment='accounts cleaned and bad data')
def customer_clean():
    df_customer = dlt.readStream('accounts_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=true")
        .drop("is_quarantined")
    )

dlt.create_streaming_table(
        name="dim_accounts",
        comment="accounts dim table with SCD2 approach",
        table_properties={"quality": "silver"}
        )
dlt.apply_changes(
        target = "dim_accounts",
        source = "accounts_clean_good_records",
        keys = ["account_id","customer_id"],
        sequence_by = col("snapshot_date"),
        ignore_null_updates=False,
        stored_as_scd_type="2",
        track_history_except_column_list=['snapshot_date']
    )

#------------------------------- CARD TRANSACTIONS -------------------------
@dlt.table(
    name="card_transactions_silver",
    comment="This table contains the cleaned card transactions data",
    table_properties={"quality": "silver"}
)
def card_transactions_silver():
    df_ct = dlt.readStream(f"{catalog}.{bronze_schema}.card_transactions_bronze")
    return(
        df_ct
        .withColumn("amount", col("amount").cast(DecimalType(20,2)))
        .withColumn("datum", col("datum").cast(DateType()))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- CARDS  -------------------------
cards_expectations = {
    "valid_card":"card_id is not null and account_id is not null and typ_karty is not null",
}

cards_expectations_expr = "NOT({0})".format(" AND ".join(cards_expectations.values()))

@dlt.table(
    name = "cards_clean_silver",
    comment = "This table contains the cleaned cards data",
    partition_cols =["is_quarantined"]
)
@dlt.expect_all(cards_expectations)
def cards_clean_silver():
    df_cards = dlt.readStream(f"{catalog}.{bronze_schema}.cards_bronze")
    return (
        df_cards
        .withColumn("platnost_do",col("platnost_do").cast(DateType()))
        .withColumn("is_quarantined",expr(cards_expectations_expr))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

@dlt.table(name='cards_clean_good_records',comment='cards cleanded and validate data')
def customer_clean():
    df_customer = dlt.readStream('cards_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=false")
        .drop("is_quarantined")
    )

@dlt.table(name='cards_clean_bad_records',comment='cards cleaned and bad data')
def customer_clean():
    df_customer = dlt.readStream('cards_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=true")
        .drop("is_quarantined")
    )

dlt.create_streaming_table(
        name="dim_cards",
        comment="cards dim table with SCD2 approach",
        table_properties={"quality": "silver"}
        )
dlt.apply_changes(
        target = "dim_cards",
        source = "cards_clean_good_records",
        keys = ["card_id","account_id"],
        sequence_by = col("snapshot_date"),
        ignore_null_updates=False,
        stored_as_scd_type="2",
        track_history_except_column_list=['snapshot_date']
    )


#------------------------------- EMPLOYEES  -------------------------
employees_expectations = {
    "valid_employee":"employee_id is not null and prijmeni is not null and role is not null",
}

employees_expectations_expr = "NOT({0})".format(" AND ".join(employees_expectations.values()))

@dlt.table(
    name = "employees_clean_silver",
    comment = "This table contains the cleaned employees data",
    partition_cols =["is_quarantined"]
)
@dlt.expect_all(employees_expectations)
def employees_clean_silver():
    df_employees = dlt.readStream(f"{catalog}.{bronze_schema}.employees_bronze")
    return (
        df_employees
        .withColumn("is_quarantined",expr(employees_expectations_expr))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

@dlt.table(name='employees_clean_good_records',comment='employees cleanded and validate data')
def customer_clean():
    df_customer = dlt.readStream('employees_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=false")
        .drop("is_quarantined")
    )

@dlt.table(name='employees_clean_bad_records',comment='employees cleaned and bad data')
def customer_clean():
    df_customer = dlt.readStream('employees_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=true")
        .drop("is_quarantined")
    )

dlt.create_streaming_table(
        name="dim_employees",
        comment="employees dim table with SCD2 approach",
        table_properties={"quality": "silver"}
        )
dlt.apply_changes(
        target = "dim_employees",
        source = "employees_clean_good_records",
        keys = ["employee_id"],
        sequence_by = col("snapshot_date"),
        ignore_null_updates=False,
        stored_as_scd_type="2",
        track_history_except_column_list=['snapshot_date']
    )

#------------------------------- FINANCIAL TRANSACTIONS  -------------------------
@dlt.table(
    name="financial_transactions_silver",
    comment="This table contains the cleaned financial transactions data",
    table_properties={"quality": "silver"}
)
def fiancial_transactions_silver():
    df_ft = dlt.readStream(f"{catalog}.{bronze_schema}.financial_transactions_bronze")
    return(
        df_ft
        .withColumn("amount", col("amount").cast(DecimalType(20,2)))
        .withColumn("datum", col("datum").cast(DateType()))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- MERCHATS  -------------------------
@dlt.table(
    name="merchants_silver",
    comment="This table contains the cleaned merchants data",
    table_properties={"quality": "silver"}
)
def merchants_silver():
    df_m = dlt.read(f"{catalog}.{bronze_schema}.merchants_bronze")
    return(
        df_m
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- PRODUCTS  -------------------------
@dlt.table(
    name="products_silver",
    comment="This table contains the cleaned products data",
    table_properties={"quality": "silver"}
)
def products_silver():
    df_p = dlt.read(f"{catalog}.{bronze_schema}.products_bronze")
    return(
        df_p
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- REGIONS  -------------------------
@dlt.table(
    name="regions_silver",
    comment="This table contains the cleaned regions data",
    table_properties={"quality": "silver"}
)
def region_silver():
    df_p = dlt.read(f"{catalog}.{bronze_schema}.regions_bronze")
    return(
        df_p
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- CITIES  -------------------------
@dlt.table(
    name="cities_silver",
    comment="This table contains the cleaned cities data",
    table_properties={"quality": "silver"}
)
def cities_silver():
    df_c = dlt.read(f"{catalog}.{bronze_schema}.cities_bronze")
    return(
        df_c
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- BRANCHES  -------------------------
@dlt.table(
    name="branches_silver",
    comment="This table contains the cleaned branches data",
    table_properties={"quality": "silver"}
)
def branches_silver():
    cp_reg = r'(\d+[A-Za-z]?(\s*/\s*\d+[A-Za-z]?)?)\s*$'
    df_b = dlt.read(f"{catalog}.{bronze_schema}.branches_bronze")
    return (
        df_b
        .withColumn("zip_code", split(col("adresa"), ",").getItem(2))
        .withColumn("mesto", split(col("adresa"), ",").getItem(1))
        .withColumn("_ulice", split(col("adresa"), ",").getItem(0))
        .withColumn("cislo_popisne", trim(regexp_extract(col("_ulice"), cp_reg, 1)))
        .withColumn("ulice", trim(regexp_replace(col("_ulice"), cp_reg, "")))
        .drop("_ulice","_rescued_data","source_file","ingestion_ts")
)